In [1]:
import boto3
from botocore.client import Config

s3 = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="minioadmin",
    aws_secret_access_key="minioadmin123",
    region_name="us-east-1",
    config=Config(signature_version="s3v4", s3={"addressing_style": "path"}),
)

s3.upload_file("/home/work/data/transactions_history.jsonl", "lake", "bronze/history/transactions_history.jsonl")
s3.upload_file("/home/work/merchants.json", "lake", "reference/merchants.json")

for o in s3.list_objects_v2(Bucket="lake").get("Contents", []):
    print(o["Key"], o["Size"])

bronze/history/transactions_history.jsonl 11641956
reference/merchants.json 45181


In [1]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("SahlPay-Partie2-RDD")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.executor.memory", "1g")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate())

sc = spark.sparkContext
sc.setLogLevel("WARN")

rdd = sc.textFile("s3a://lake/bronze/history/transactions_history.jsonl")
print("Lignes      :", rdd.count())
print("Partitions  :", rdd.getNumPartitions())
print(rdd.take(2))

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a5dead2c-fe12-4529-9375-f74613a14bde;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
downloading https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.3.4/hadoop-aws-3.3.4.jar ...
	[SUCCESSFUL ] org.apache.hadoop#hadoop-aws;3.3.4!hadoop-aws.jar (308ms)
downloading https://repo1.maven.org/maven2/com/amazonaws/aws-java-sdk-bundle/1.12.262/aws-java-sdk-bundle-1.12.262.jar ...
	[SUCCESSFUL ] com.amazonaws#aws-java-sdk-bundle;1.12.262!aws-java-sdk-bundle.jar (25434ms)
downloading https://repo1.maven.org/maven2/org/wildfly/openssl/wildfly-openssl/1.0.7.Final/wildfly-

Lignes      : 50000
Partitions  : 2


[Stage 1:>                                                          (0 + 1) / 1]

['{"tx_id": "TX-00000001", "type": "P2M", "payer_wallet": "W-10481", "payee_id": "M-100", "device_id": "D-c9dc72", "city": "Rabat", "channel": "QR", "status": "INITIATED", "amount_mad": 71.09, "event_time": "2026-11-14T08:00:02Z"}', '{"tx_id": "TX-00000002", "type": "P2M", "payer_wallet": "W-10972", "payee_id": "M-261", "device_id": "D-0ed2ec", "city": "Meknes", "channel": "APP", "status": "INITIATED", "amount_mad": 182.07, "event_time": "2026-11-14T08:00:04Z"}']


In [2]:
import json, time
from operator import add

t_debut = time.time()

lines  = sc.textFile("s3a://lake/bronze/history/transactions_history.jsonl")
events = lines.map(json.loads)                                        # parse chaque ligne
completed_raw = events.filter(lambda e: e["status"] == "COMPLETED")   # garde COMPLETED

print("Événements         :", events.count())
print("COMPLETED (brut)   :", completed_raw.count())

Événements         : 50000
COMPLETED (brut)   : 15697


In [3]:
completed = (completed_raw
    .map(lambda e: (e["tx_id"], e))
    .reduceByKey(lambda a, b: a)      # un seul événement par tx_id
    .values())

print("COMPLETED (dédoublonné) :", completed.count())

[Stage 4:>                                                          (0 + 2) / 2]

COMPLETED (dédoublonné) : 15401


In [4]:
total_par_ville = (completed
    .map(lambda e: (e["city"], e["amount_mad"]))
    .reduceByKey(add))

for ville, total in sorted(total_par_ville.collect(), key=lambda x: -x[1]):
    print(f"{ville:12s} {total:14,.2f} MAD")

Casablanca       969,822.08 MAD
Rabat            577,769.98 MAD
Marrakech        501,856.70 MAD
Fes              409,119.30 MAD
Tanger           383,106.01 MAD
Agadir           333,253.03 MAD
Meknes           262,511.07 MAD
Tetouan          227,248.18 MAD
Kenitra          198,912.47 MAD
Oujda            157,724.07 MAD


In [5]:
somme_nb = (completed
    .map(lambda e: (e["city"], (e["amount_mad"], 1)))
    .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])))

moyenne_par_ville = somme_nb.mapValues(lambda v: (v[0], v[1], v[0] / v[1]))

print(f"{'Ville':12s} {'Total (MAD)':>14s} {'Nb':>7s} {'Moyenne (MAD)':>14s}")
for ville, (tot, nb, moy) in sorted(moyenne_par_ville.collect(), key=lambda x: -x[1][0]):
    print(f"{ville:12s} {tot:14,.2f} {nb:7d} {moy:14,.2f}")

Ville           Total (MAD)      Nb  Moyenne (MAD)
Casablanca       969,822.08    3740         259.31
Rabat            577,769.98    2180         265.03
Marrakech        501,856.70    1950         257.36
Fes              409,119.30    1520         269.16
Tanger           383,106.01    1511         253.54
Agadir           333,253.03    1274         261.58
Meknes           262,511.07    1016         258.38
Tetouan          227,248.18     766         296.67
Kenitra          198,912.47     758         262.42
Oujda            157,724.07     686         229.92


In [6]:
top5 = (completed
    .filter(lambda e: e["type"] == "P2M")
    .map(lambda e: (e["payee_id"], 1))
    .reduceByKey(add)
    .takeOrdered(5, key=lambda x: -x[1]))

noms = {}
with open("/home/work/merchants.json", encoding="utf-8") as f:
    for l in f:
        m = json.loads(l)
        noms[m["merchant_id"]] = (m["name"], m["category"])

for mid, nb in top5:
    print(mid, noms[mid][0], f"({noms[mid][1]})", "->", nb, "paiements")

print("Durée totale du notebook (s) :", round(time.time() - t_debut, 1))

M-107 Pharmacie Ibn Sina Marrakech 2 (pharmacie) -> 373 paiements
M-240 Epicerie Al Baraka Agadir 2 (epicerie) -> 293 paiements
M-090 Pharmacie du Marche Meknes (pharmacie) -> 204 paiements
M-033 Restaurant Al Medina Fes (restauration) -> 202 paiements
M-038 Epicerie Al Amal Marrakech 2 (epicerie) -> 199 paiements
Durée totale du notebook (s) : 161.6


In [7]:
print(moyenne_par_ville.toDebugString().decode("utf-8"))

(2) PythonRDD[22] at collect at /tmp/ipykernel_75/3597887765.py:8 []
 |  MapPartitionsRDD[21] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[20] at partitionBy at <unknown>:0 []
 +-(2) PairwiseRDD[19] at reduceByKey at /tmp/ipykernel_75/3597887765.py:1 []
    |  PythonRDD[18] at reduceByKey at /tmp/ipykernel_75/3597887765.py:1 []
    |  MapPartitionsRDD[11] at mapPartitions at PythonRDD.scala:160 []
    |  ShuffledRDD[10] at partitionBy at <unknown>:0 []
    +-(2) PairwiseRDD[9] at reduceByKey at /tmp/ipykernel_75/755411122.py:1 []
       |  PythonRDD[8] at reduceByKey at /tmp/ipykernel_75/755411122.py:1 []
       |  s3a://lake/bronze/history/transactions_history.jsonl MapPartitionsRDD[5] at textFile at <unknown>:0 []
       |  s3a://lake/bronze/history/transactions_history.jsonl HadoopRDD[4] at textFile at <unknown>:0 []


In [9]:
spark.stop()